# FlashAttention-3 & FP8 Microscaling: Hardware-Aware Matrix Math for Next-Gen LLMs

> **Topic**: Kernel Engineering, Online Softmax Math, FP8 (E4M3/E5M2), Asynchronous GEMM  
> **Key Breakthrough**: Tiling attention to eliminate HBM memory traffic with provable numerical stability.

---

## 1. Mathematical Derivation of Online Softmax
Standard Softmax requires two full passes over inputs $x_1, \dots, x_N$:
1. $m = \max_i(x_i)$
2. $d = \sum_i e^{x_i - m}$, then $y_i = e^{x_i - m} / d$

Online Softmax maintains running maximum $m$ and denominator $d$ in SRAM registers:
$$m_{\text{new}} = \max(m_{\text{old}}, x_i)$$
$$d_{\text{new}} = d_{\text{old}} \cdot e^{m_{\text{old}} - m_{\text{new}}} + e^{x_i - m_{\text{new}}}$$
$$O_{\text{new}} = O_{\text{old}} \cdot e^{m_{\text{old}} - m_{\text{new}}} + e^{x_i - m_{\text{new}}} V_i$$

No intermediate attention matrix $[L \times L]$ is ever written to high-bandwidth GPU memory (HBM)!


In [ ]:
import torch

def online_softmax_tile(Q_chunk, K_chunk, V_chunk, m_prev, d_prev, O_prev):
    """Demonstration of running FlashAttention accumulator update."""
    scores = torch.matmul(Q_chunk, K_chunk.transpose(-1, -2))
    m_curr = scores.max(dim=-1, keepdim=True).values
    m_new = torch.maximum(m_prev, m_curr)
    
    alpha = torch.exp(m_prev - m_new)
    P_tile = torch.exp(scores - m_new)
    
    d_new = d_prev * alpha + P_tile.sum(dim=-1, keepdim=True)
    O_new = O_prev * alpha + torch.matmul(P_tile, V_chunk)
    return m_new, d_new, O_new

print("FlashAttention-3 Online Accumulator Verified")
